In [4]:
import os
import cv2
import time
import json
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
from scipy.signal import wiener
from skimage.util import random_noise
from skimage.metrics import mean_squared_error as calc_mse
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

# ==========================================
# 1. NOISE ADDITION
# ==========================================
def add_custom_noise(image, modality):
    if modality.lower() == 'ultrasound':
        noisy = random_noise(image, mode='speckle', var=0.05)
    else:
        noisy = random_noise(image, mode='gaussian', var=0.01)
    return (noisy * 255).astype(np.uint8)

# ==========================================
# 2. FILTER IMPLEMENTATIONS
# ==========================================
def filter_mean(img): return cv2.blur(img, (5, 5))
def filter_gaussian(img): return cv2.GaussianBlur(img, (5, 5), 1.5)
def filter_median(img): return cv2.medianBlur(img, 5)
def filter_hybrid(img): return filter_mean(filter_median(img))
def filter_moving_average(img): return cv2.filter2D(img, -1, np.ones((1, 5), np.float32) / 5)

def filter_ema(img):
    res = np.zeros_like(img, dtype=np.float32)
    res[:, 0] = img[:, 0]
    for i in range(1, img.shape[1]):
        res[:, i] = 0.3 * img[:, i] + 0.7 * res[:, i-1]
    return res.astype(np.uint8)

def filter_bilateral(img): return cv2.bilateralFilter(img, 9, 75, 75)

def filter_anisotropic_diffusion(img, niter=10, kappa=50, gamma=0.1):
    img = img.astype('float32')
    for _ in range(niter):
        deltaD = np.zeros_like(img); deltaD[1:, :] = np.diff(img, axis=0)
        deltaS = np.zeros_like(img); deltaS[:-1, :] = -np.diff(img, axis=0)
        deltaE = np.zeros_like(img); deltaE[:, 1:] = np.diff(img, axis=1)
        deltaW = np.zeros_like(img); deltaW[:, :-1] = -np.diff(img, axis=1)
        img += gamma * (np.exp(-(deltaD/kappa)**2)*deltaD + np.exp(-(deltaS/kappa)**2)*deltaS + 
                        np.exp(-(deltaE/kappa)**2)*deltaE + np.exp(-(deltaW/kappa)**2)*deltaW)
    return np.clip(img, 0, 255).astype(np.uint8)

def filter_nlm(img): return cv2.fastNlMeansDenoising(img, None, 10, 7, 21)

def filter_wiener(img): return np.clip(wiener(img.astype(float), (5, 5)), 0, 255).astype(np.uint8)

def filter_wavelet(img):
    LL, (LH, HL, HH) = pywt.dwt2(img, 'haar')
    LH, HL, HH = [pywt.threshold(x, 10, mode='soft') for x in (LH, HL, HH)]
    reconstructed = pywt.idwt2((LL, (LH, HL, HH)), 'haar')
    
    # Crop to handle odd-sized image dimensions causing shape mismatch
    reconstructed = reconstructed[:img.shape[0], :img.shape[1]]
    return np.clip(reconstructed, 0, 255).astype(np.uint8)

def filter_curvelet(img): return img # Placeholder: requires external C++ wrapper

FILTERS = {
    'Mean': filter_mean, 'Moving_Avg': filter_moving_average, 'EMA': filter_ema,
    'Gaussian': filter_gaussian, 'Median': filter_median, 'Hybrid': filter_hybrid,
    'Bilateral': filter_bilateral, 'Anisotropic': filter_anisotropic_diffusion, 'NLM': filter_nlm,
    'Wiener': filter_wiener, 'Wavelet': filter_wavelet, 'Curvelet': filter_curvelet
}

# ==========================================
# 3. METRICS EVALUATION
# ==========================================
def evaluate(clean, denoised):
    mse = calc_mse(clean, denoised)
    psnr = calc_psnr(clean, denoised, data_range=255)
    ssim = calc_ssim(clean, denoised, data_range=255)
    noise_power = np.mean((clean.astype(float) - denoised.astype(float)) ** 2)
    snr = float('inf') if noise_power == 0 else 10 * np.log10(np.mean(clean.astype(float)**2) / noise_power)
    return mse, psnr, ssim, snr

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================

base_out_dir = "OUTPUT_RESULTS"
modalities = ["ct", "MRI", "Ultrasound"]
results = []

for mod in modalities:
    in_mod_path = os.path.join( mod)
    if not os.path.exists(in_mod_path): 
        print(f"Directory not found: {in_mod_path}")
        continue
        
    corrupted_dir = os.path.join(base_out_dir, "Corrupted", mod)
    os.makedirs(corrupted_dir, exist_ok=True)
    
    images = [f for f in os.listdir(in_mod_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
    
    for img_name in images:
        img_path = os.path.join(in_mod_path, img_name)
        clean_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if clean_img is None: continue
        
        noisy_img = add_custom_noise(clean_img, mod)
        cv2.imwrite(os.path.join(corrupted_dir, img_name), noisy_img)
        
        for f_name, f_func in FILTERS.items():
            denoised_dir = os.path.join(base_out_dir, "Denoised", mod, f_name)
            os.makedirs(denoised_dir, exist_ok=True)
            
            start_time = time.time()
            denoised_img = f_func(noisy_img)
            exec_time = time.time() - start_time
            
            cv2.imwrite(os.path.join(denoised_dir, img_name), denoised_img)
            
            mse, psnr, ssim, snr = evaluate(clean_img, denoised_img)
            results.append({
                'Modality': mod, 'Image': img_name, 'Filter': f_name,
                'MSE': mse, 'PSNR': psnr, 'SSIM': ssim, 'SNR': snr, 'Time(s)': exec_time
            })

# ==========================================
# 5. DATA EXPORT & PLOTTING
# ==========================================
if not results:
    print("No results generated. Check input folder structure.")
else:
    df = pd.DataFrame(results)

    # Save raw results to JSON
    with open(os.path.join(base_out_dir, "metrics_full.json"), "w") as f:
        json.dump(results, f, indent=4)

    # Aggregate and save summary
    summary_df = df.groupby(['Modality', 'Filter']).mean(numeric_only=True).reset_index()
    summary_df.to_json(os.path.join(base_out_dir, "metrics_summary.json"), orient="records", indent=4)
    summary_df.to_csv(os.path.join(base_out_dir, "metrics_summary.csv"), index=False)

    # Generate Plots
    metrics = ['MSE', 'PSNR', 'SSIM', 'SNR']
    plots_dir = os.path.join(base_out_dir, "Plots")
    os.makedirs(plots_dir, exist_ok=True)

    for metric in metrics:
        plt.figure(figsize=(14, 8))
        for i, mod in enumerate(modalities):
            mod_data = summary_df[summary_df['Modality'] == mod]
            if mod_data.empty: continue
            
            plt.subplot(3, 1, i+1)
            plt.bar(mod_data['Filter'], mod_data[metric], color='skyblue')
            plt.title(f"{metric} Comparison for {mod.upper()}")
            plt.ylabel(metric)
            plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f"{metric}_comparison.png"), dpi=300)
        plt.close()

    print(f"Pipeline complete. All images, metrics (JSON/CSV), and plots saved in '{base_out_dir}'.")

Pipeline complete. All images, metrics (JSON/CSV), and plots saved in 'OUTPUT_RESULTS'.
